# 🏈 Go Blue: A Pandas Data Exploration

### SIADS 505 · Other Assignments · Assignment 1
**100 points — 10 questions × 10 points each**

---

It is a cold Saturday in Ann Arbor. 110,000 people are yelling. You are in the
press box with a laptop and a CSV file, and somebody upstairs wants answers
before the fourth quarter.

This assignment walks you through a full data exploration of **fifteen seasons
of Michigan football** — loading, cleaning, filtering, grouping, merging, and
pivoting — using nothing but `pandas` and ordinary Python. No machine learning,
no fancy libraries. Just the fundamentals, which is what actually gets used at
3 a.m. when the data is wrong and the deadline is real.

> ### ⚠️ About this data
> The dataset is **simulated**. The team names, conferences, stadiums, and the
> coaching timeline are real; every score, attendance figure, temperature, and
> ticket price was generated by `tools/generate_data.py` from a fixed random
> seed. Please do not cite anything in here as a real Michigan statistic — the
> Wolverines did not actually go 4–11 against Ohio State. Enjoy the fiction.

---

## How this is graded

Each question asks you to **write one function**. Underneath each one you'll
find a cell of **public asserts** — run them as often as you like, they are
your smoke test and they are worth **zero points**.

Your grade comes from a set of **hidden asserts** you do not get to see. They
check the same functions against the same rules, plus a few extra cases —
including running your function on **a different slice of the data**. So:

- ✅ Compute your answer *from the DataFrame you are handed*.
- ❌ Do not hard-code the values you saw in the public asserts. It will not work.

Each question is worth **10 points**, awarded in parts, so partial credit is
real. Passing the public asserts is a good sign but it is not a guarantee.

**Rules of the road**

1. Only edit inside the function bodies marked `# YOUR CODE HERE`.
2. Don't rename the functions or change their parameters.
3. Every function takes its data as an argument and **returns** a value.
   Nothing is graded by printing.
4. Don't modify a DataFrame you were handed — work on a `.copy()`.
5. `data/games.csv` and `data/opponents.csv` are read-only. Leave them alone.


## The data

Two CSV files live in `data/`.

### `data/games.csv` — one row per game, 195 rows

| column | what it is |
|---|---|
| `game_id` | unique id, e.g. `2016-W07` (season + week) |
| `season` | 2010 through 2024 |
| `week` | 1–13 |
| `date` | game date, `YYYY-MM-DD` |
| `kickoff` | local kickoff time, e.g. `3:30 PM` |
| `opponent` | who Michigan played |
| `site` | `Home`, `Away`, or `Neutral` |
| `stadium` | where it was played |
| `michigan_points` | final Michigan score |
| `opponent_points` | final opponent score |
| `attendance` | tickets scanned — **has gaps** |
| `temperature_f` | kickoff temperature — **has gaps** |
| `weather` | `Sunny`, `Cloudy`, `Overcast`, `Rain`, `Snow`, `Windy` |
| `student_tickets_sold` | home games only, so it is empty for road games |
| `avg_ticket_price` | resale average — **a messy string**, e.g. `"$212.40"` |
| `head_coach` | who was in charge that season |

### `data/opponents.csv` — one row per team, 32 rows

| column | what it is |
|---|---|
| `opponent` | team name — this is the key that joins to `games.csv` |
| `conference` | `Big Ten`, `SEC`, `MAC`, … |
| `mascot` | Buckeyes, Spartans, Boilermakers, … |
| `home_state` | two-letter state code |
| `home_stadium` | their stadium |
| `stadium_capacity` | seats |
| `founded` | year the school was founded |
| `is_rival` | `True` for the three teams that make the schedule personal |

Heads up: `opponents.csv` lists **more teams than Michigan actually played**.
A join has to be careful about that.


---

## Setup

Run this first.

In [ ]:
import numpy as np
import pandas as pd

# A little breathing room when you print a frame.
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

print(f"pandas {pd.__version__} · numpy {np.__version__}")
print("Let's go.")

### First contact

Before writing anything, look at the thing. Run the cell below and scroll. This
one is ungraded — poke at it, add cells, break things. That is the job.


In [ ]:
raw = pd.read_csv("data/games.csv")

print(raw.shape)
display(raw.head())
display(raw.dtypes.to_frame("dtype"))
display(raw.isna().sum().to_frame("missing values").query("`missing values` > 0"))

---

## Question 1 — Kickoff  ·  10 points

Everything starts with `read_csv`, and almost everyone gets it slightly wrong
the first time.

Write **`load_games(path="data/games.csv")`** so that it:

1. reads the CSV at `path`,
2. parses the **`date`** column as a real datetime (not a string), and
3. sets **`game_id`** as the index.

It should return a `DataFrame` with **195 rows and 15 columns** — 15, not 16,
because `game_id` moved out of the columns and into the index.

*Hint:* `pd.read_csv` takes a `parse_dates=` argument, and `DataFrame` has a
`.set_index()` method.


In [ ]:
def load_games(path="data/games.csv"):
    """Load games.csv with a datetime `date` column, indexed by `game_id`."""
    games = pd.read_csv(path, parse_dates=["date"])
    return games.set_index("game_id")

**Public checks — worth 0 points. Run them.**

In [ ]:
games = load_games()

assert isinstance(games, pd.DataFrame), "load_games should return a DataFrame"
assert games.shape == (195, 15), f"expected (195, 15), got {games.shape}"
assert games.index.name == "game_id", "game_id should be the index"
assert pd.api.types.is_datetime64_any_dtype(games["date"]), "`date` should be datetime"
assert games.loc["2016-W07", "season"] == 2016

print("✅ Q1 public checks passed — you are on the field.")
games.head()

---

## Question 2 — Scoreboard math  ·  10 points

A raw score is two numbers. What people actually argue about is the *margin*.

Write **`add_result_columns(games)`** which takes the games frame and returns a
**new** DataFrame with every original column plus three more:

| new column | definition |
|---|---|
| `point_diff` | `michigan_points - opponent_points` (negative on a loss) |
| `total_points` | `michigan_points + opponent_points` |
| `won` | `True` when Michigan scored more, else `False` — a real boolean |

**The frame you were handed must come back unchanged.** Start with
`games.copy()`, add columns to the copy, return the copy. This is the single
most common way to break a pandas notebook, and the hidden tests check for it.


In [ ]:
def add_result_columns(games):
    """Return a copy of `games` with point_diff, total_points and won added."""
    out = games.copy()
    out["point_diff"] = out["michigan_points"] - out["opponent_points"]
    out["total_points"] = out["michigan_points"] + out["opponent_points"]
    out["won"] = out["michigan_points"] > out["opponent_points"]
    return out

**Public checks — worth 0 points. Run them.**

In [ ]:
scored = add_result_columns(games)

for col in ["point_diff", "total_points", "won"]:
    assert col in scored.columns, f"missing column: {col}"
assert scored["won"].dtype == bool, "`won` should be True/False, not 1/0 or 'Yes'"
assert scored.shape[0] == 195
assert "point_diff" not in games.columns, "you modified the original frame — use .copy()"

row = scored.loc["2010-W01"]
assert row["point_diff"] == row["michigan_points"] - row["opponent_points"]
assert row["total_points"] == row["michigan_points"] + row["opponent_points"]

print(f"✅ Q2 public checks passed — simulated all-time record: "
      f"{scored['won'].sum()}-{(~scored['won']).sum()}")
scored[["opponent", "michigan_points", "opponent_points", "point_diff", "won"]].head()

---

## Question 3 — The statement win and the long bus ride home  ·  10 points

Every fan base remembers exactly two games: the one they still talk about, and
the one nobody mentions.

Write **`best_and_worst(games)`** that returns a **tuple of two `game_id`
strings**:

```python
(game_id_of_biggest_win, game_id_of_worst_loss)
```

- The **biggest win** is the game with the largest `michigan_points - opponent_points`.
- The **worst loss** is the game with the smallest (most negative) margin.

*Hint:* build the margin as a Series, then reach for `.idxmax()` and
`.idxmin()` — they hand you the **index label**, which is exactly the
`game_id` you want. Sorting and grabbing row zero also works.

There are no ties in this dataset, so you don't have to break one.


In [ ]:
def best_and_worst(games):
    """Return (biggest_win_game_id, worst_loss_game_id)."""
    diff = games["michigan_points"] - games["opponent_points"]
    return (diff.idxmax(), diff.idxmin())

**Public checks — worth 0 points. Run them.**

In [ ]:
best, worst = best_and_worst(games)

assert isinstance(best_and_worst(games), tuple), "return a tuple"
assert isinstance(best, str) and isinstance(worst, str), "return game_id strings"
assert best in games.index and worst in games.index

b, w = games.loc[best], games.loc[worst]
assert b["michigan_points"] - b["opponent_points"] == 61
assert w["michigan_points"] - w["opponent_points"] == -43

print(f"🎉 Biggest win : {best} — Michigan {b['michigan_points']}, "
      f"{b['opponent']} {b['opponent_points']}")
print(f"😬 Worst loss  : {worst} — Michigan {w['michigan_points']}, "
      f"{w['opponent']} {w['opponent_points']}")
print("✅ Q3 public checks passed.")

---

## Question 4 — Follow the money  ·  10 points

Somebody exported the resale ticket data straight out of the box office system,
and it shows. `avg_ticket_price` is text: it has dollar signs, a few rows
picked up stray whitespace, and five games the box office never reported at all
— those read **`"unavailable"`**.

Run this to see the damage:

```python
games["avg_ticket_price"].unique()[:10]
```

Write **`ticket_prices(games)`** that returns a `float` **Series**:

- named **`ticket_price`**,
- with the **same index and the same 195 rows** as `games`,
- with `"unavailable"` turned into **`NaN`** (not `0`, not dropped),
- and everything else as a number: `"  $212.40 "` → `212.4`.

*Hint:* the `.str` accessor gives you `.strip()` and `.replace()` on a whole
column at once. `.str.replace("$", "", regex=False)` avoids a nasty surprise,
because in a regular expression `$` means "end of string". Then let
`pd.to_numeric(..., errors="coerce")` turn whatever is left over into `NaN`.


In [ ]:
def ticket_prices(games):
    """Return avg_ticket_price as a float Series named 'ticket_price'."""
    cleaned = (
        games["avg_ticket_price"]
        .astype(str)
        .str.strip()
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
    )
    prices = pd.to_numeric(cleaned, errors="coerce")
    prices.name = "ticket_price"
    return prices

**Public checks — worth 0 points. Run them.**

In [ ]:
prices = ticket_prices(games)

assert isinstance(prices, pd.Series), "return a Series"
assert prices.name == "ticket_price", f"name it 'ticket_price', got {prices.name!r}"
assert pd.api.types.is_float_dtype(prices), f"expected floats, got {prices.dtype}"
assert len(prices) == 195, "keep all 195 rows — coerce, don't drop"
assert prices.isna().sum() == 5, "the 5 'unavailable' games should be NaN"
assert abs(prices.max() - 433.59) < 0.01
assert abs(prices.mean() - 166.11) < 0.01   # .mean() skips NaN for you

pricey = prices.nlargest(3)
print("💸 Most expensive tickets in the simulation:")
for gid, price in pricey.items():
    print(f"   ${price:,.2f}  {gid}  vs {games.loc[gid, 'opponent']}")
print("✅ Q4 public checks passed.")

---

## Question 5 — The season report card  ·  10 points

Time for `groupby`, the single most useful verb in pandas.

Write **`season_report_card(games)`** returning a DataFrame with **one row per
season**, sorted oldest to newest:

- index: `season`, and the index should be **named** `season`
- columns, **in exactly this order**:

| column | definition |
|---|---|
| `wins` | games where Michigan outscored the opponent |
| `losses` | everything else (there are no ties) |
| `points_for` | total `michigan_points` that season |
| `points_against` | total `opponent_points` that season |
| `win_pct` | `wins / (wins + losses)`, **rounded to 3 decimals** |

*Hint:* a boolean Series sums like a counter — `True` is 1. So
`df.groupby("season")["won"].sum()` counts wins, and `.size()` counts games.
Build the pieces, then stitch them together with `pd.DataFrame({...})`.


In [ ]:
def season_report_card(games):
    """One row per season: wins, losses, points_for, points_against, win_pct."""
    won = games["michigan_points"] > games["opponent_points"]
    grouped = games.assign(_won=won).groupby("season")
    out = pd.DataFrame({
        "wins": grouped["_won"].sum().astype(int),
        "losses": (grouped.size() - grouped["_won"].sum()).astype(int),
        "points_for": grouped["michigan_points"].sum().astype(int),
        "points_against": grouped["opponent_points"].sum().astype(int),
    })
    out["win_pct"] = (out["wins"] / (out["wins"] + out["losses"])).round(3)
    return out.sort_index()

**Public checks — worth 0 points. Run them.**

In [ ]:
card = season_report_card(games)

assert list(card.columns) == ["wins", "losses", "points_for", "points_against", "win_pct"]
assert card.index.name == "season"
assert list(card.index) == list(range(2010, 2025)), "seasons 2010..2024, ascending"
assert (card["wins"] + card["losses"] == 13).all(), "13 games every season"
assert card.loc[2023, "wins"] == 13 and card.loc[2023, "losses"] == 0
assert abs(card.loc[2020, "win_pct"] - 0.462) < 1e-6

print("🏆 Undefeated seasons:", list(card.index[card["win_pct"] == 1.0]))
print("✅ Q5 public checks passed.")
card

---

## Question 6 — Under the lights  ·  10 points

Sports radio will tell you Michigan is a different team at night. Sports radio
tells you a lot of things. Let's check.

Write **`night_game_split(games)`** returning a **Series** indexed by
`["Day", "Night"]` **in that order**, whose values are the **win percentage**
for each group, **rounded to 3 decimals**.

A game is a **night game** if it kicked off at **6:00 PM or later**.

The catch: `kickoff` is a string like `"3:30 PM"` or `"7:00 PM"`. You need the
hour as a number, and `"12:00 PM"` has to come out as **noon**, not midnight.

*Hint:* `pd.to_datetime(games["kickoff"], format="%I:%M %p")` parses a
12-hour clock correctly. Then `.dt.hour` gives you an integer, and
`np.where(hour >= 18, "Night", "Day")` labels each row. Group the boolean
"did we win" Series by those labels and take `.mean()` — the mean of a boolean
*is* the win percentage.


In [ ]:
def night_game_split(games):
    """Win pct for day vs night kickoffs, as a Series indexed ['Day', 'Night']."""
    hour = pd.to_datetime(games["kickoff"], format="%I:%M %p").dt.hour
    label = np.where(hour >= 18, "Night", "Day")
    won = games["michigan_points"] > games["opponent_points"]
    pct = won.groupby(label).mean().round(3)
    pct = pct.reindex(["Day", "Night"])
    pct.name = "win_pct"
    pct.index.name = "time_of_day"
    return pct

**Public checks — worth 0 points. Run them.**

In [ ]:
split = night_game_split(games)

assert isinstance(split, pd.Series), "return a Series"
assert list(split.index) == ["Day", "Night"], f"index should be ['Day', 'Night'], got {list(split.index)}"
assert 0 <= split.min() and split.max() <= 1, "these are proportions, not percentages out of 100"
assert abs(split["Day"] - 0.717) < 1e-6
assert abs(split["Night"] - 0.719) < 1e-6

print(f"☀️  Day   {split['Day']:.1%}")
print(f"🌙 Night  {split['Night']:.1%}")
print("...two tenths of one percent. Sports radio remains undefeated at saying things.")
print("✅ Q6 public checks passed.")

---

## Question 7 — Counting the crowd  ·  10 points

Thirteen games are missing an `attendance` number. You cannot drop them —
they're real games — and you cannot fill them with zero, which would claim
nobody showed up to a sold-out stadium.

The sane move is to fill each gap with the **median attendance of games at the
same `site`**. A missing home game gets the median home crowd (~109,000); a
missing road game gets the median road crowd. Filling all thirteen with one
overall median would quietly invent 40,000 fans in Ann Arbor and lose 40,000
somewhere else.

Write **`filled_attendance(games)`** returning a `float` **Series**:

- named **`attendance`**, with the same index and all **195** rows,
- **zero** missing values,
- rows that already had a number **left exactly as they were**,
- each gap filled with the **median attendance of that game's `site` group**.

*Hint:* `groupby("site")["attendance"].transform("median")` is the trick. Unlike
`.median()`, which collapses to one value per group, `.transform()` hands back a
Series **aligned to the original rows** — perfectly shaped to pass to
`.fillna()`.


In [ ]:
def filled_attendance(games):
    """Attendance with gaps filled by the median for that game's site."""
    medians = games.groupby("site")["attendance"].transform("median")
    filled = games["attendance"].fillna(medians).astype(float)
    filled.name = "attendance"
    return filled

**Public checks — worth 0 points. Run them.**

In [ ]:
crowd = filled_attendance(games)

assert isinstance(crowd, pd.Series) and crowd.name == "attendance"
assert len(crowd) == 195 and crowd.isna().sum() == 0, "no gaps should remain"

was_missing = games["attendance"].isna()
assert (crowd[~was_missing] == games["attendance"][~was_missing]).all(), \
    "don't touch the rows that already had a number"
assert abs(crowd.mean() - 85729.17) < 0.01

by_site = games.groupby("site")["attendance"].median()
for gid in games.index[was_missing][:3]:
    site = games.loc[gid, "site"]
    assert abs(crowd[gid] - by_site[site]) < 1e-6, f"{gid} was not filled with the {site} median"

print(f"🎟️  Filled {int(was_missing.sum())} gaps.  Median crowd by site:")
print(by_site.round(0).to_string())
print("✅ Q7 public checks passed.")

---

## Question 8 — Know your enemy  ·  10 points

`games.csv` knows *who* Michigan played. `opponents.csv` knows *what conference
they were in*. Neither knows both. That is what a join is for.

Write **`record_by_conference(games, opponents)`**:

1. **Merge** `games` with `opponents` on the `opponent` column. Use a **left**
   join so that every game survives — `opponents.csv` lists teams Michigan
   never played, and an inner or right join would let them sneak in as extra
   rows. After the merge you should still have exactly **195** rows.
2. **Group** by `conference` and build a DataFrame with columns, in this order:

| column | definition |
|---|---|
| `games` | how many games Michigan played against that conference |
| `wins` | how many it won |
| `win_pct` | `wins / games`, **rounded to 3 decimals** |

3. Index by `conference` (named `conference`), and **sort by `win_pct`
   descending**, breaking ties by **conference name, A→Z**.

*Hint:* `sort_values(["win_pct", "conference"], ascending=[False, True])` sorts
by a column and by the index name in one call.


In [ ]:
def record_by_conference(games, opponents):
    """Michigan's record against each conference, best win_pct first."""
    merged = games.merge(opponents, on="opponent", how="left")
    merged["_won"] = merged["michigan_points"] > merged["opponent_points"]
    grouped = merged.groupby("conference")
    out = pd.DataFrame({
        "games": grouped.size().astype(int),
        "wins": grouped["_won"].sum().astype(int),
    })
    out["win_pct"] = (out["wins"] / out["games"]).round(3)
    return out.sort_values(["win_pct", "conference"], ascending=[False, True])

**Public checks — worth 0 points. Run them.**

In [ ]:
opponents = pd.read_csv("data/opponents.csv")
by_conf = record_by_conference(games, opponents)

assert list(by_conf.columns) == ["games", "wins", "win_pct"]
assert by_conf.index.name == "conference"
assert by_conf["games"].sum() == 195, \
    f"the merge changed the row count: {by_conf['games'].sum()} games instead of 195"
assert by_conf.index[0] == "MAC", "sorted by win_pct descending, ties A→Z"
assert by_conf.loc["Big Ten", "games"] == 135
assert abs(by_conf.loc["SEC", "win_pct"] - 0.375) < 1e-6

print("✅ Q8 public checks passed.")
by_conf

---

## Question 9 — The rivalry report  ·  10 points

Three teams on that opponents list are flagged `is_rival`. Those are the games
where the record actually matters to somebody.

Write **`rivalry_report(games, opponents)`** returning one row per rival,
indexed by `opponent` in **alphabetical order**, with columns in this order:

| column | definition |
|---|---|
| `games` | games played against that rival |
| `wins` | games won |
| `losses` | games lost |
| `win_pct` | `wins / games`, **rounded to 3 decimals** |
| `avg_point_diff` | mean of `michigan_points - opponent_points`, **rounded to 2** |

> 🚨 **Read the rival list out of `opponents`.** Do not type
> `["Ohio State", "Michigan State", "Notre Dame"]` into your function. One of
> the hidden tests hands you an opponents table with *different* teams flagged
> as rivals and checks that your answer follows. Hard-coding costs you 3 points.

*Hint:* `opponents.loc[opponents["is_rival"], "opponent"]` gives you the names,
and `games["opponent"].isin(...)` filters the schedule down to those games.


In [ ]:
def rivalry_report(games, opponents):
    """One row per rival: games, wins, losses, win_pct, avg_point_diff."""
    rivals = opponents.loc[opponents["is_rival"], "opponent"]
    played = games[games["opponent"].isin(rivals)].copy()
    played["_won"] = played["michigan_points"] > played["opponent_points"]
    played["_diff"] = played["michigan_points"] - played["opponent_points"]
    grouped = played.groupby("opponent")
    out = pd.DataFrame({
        "games": grouped.size().astype(int),
        "wins": grouped["_won"].sum().astype(int),
    })
    out["losses"] = (out["games"] - out["wins"]).astype(int)
    out["win_pct"] = (out["wins"] / out["games"]).round(3)
    out["avg_point_diff"] = grouped["_diff"].mean().round(2)
    out = out[["games", "wins", "losses", "win_pct", "avg_point_diff"]]
    return out.sort_index()

**Public checks — worth 0 points. Run them.**

In [ ]:
rivals = rivalry_report(games, opponents)

assert list(rivals.columns) == ["games", "wins", "losses", "win_pct", "avg_point_diff"]
assert list(rivals.index) == ["Michigan State", "Notre Dame", "Ohio State"], \
    "three rivals, alphabetical"
assert (rivals["wins"] + rivals["losses"] == rivals["games"]).all()
assert rivals.loc["Ohio State", "games"] == 15, "The Game is played every season"
assert abs(rivals.loc["Ohio State", "avg_point_diff"] - (-14.13)) < 0.01

print("✅ Q9 public checks passed.")
print("In this simulated timeline, The Game has not been kind. Beat Ohio State anyway.")
rivals

---

## Question 10 — Weather or not  ·  10 points

Last one. Does a Michigan November actually change how the offense scores, or
does it just make everyone in the stands miserable?

Write **`weather_scoring(games)`** returning a **pivot table**:

- **rows**: `weather`, in alphabetical order
- **columns**: `site` — `Away`, `Home`, `Neutral`
- **values**: the **mean** `michigan_points`, **rounded to 2 decimals**

Some combinations never happened — Michigan has never played a neutral-site
game in the snow. **Leave those cells as `NaN`.** An empty cell honestly says
"no games"; a `0` would lie and claim they got shut out.

*Hint:* `games.pivot_table(index=..., columns=..., values=..., aggfunc="mean")`
does the whole thing in one call, and it sorts the row labels for you.


In [ ]:
def weather_scoring(games):
    """Mean Michigan points by weather (rows) and site (columns)."""
    table = games.pivot_table(
        index="weather",
        columns="site",
        values="michigan_points",
        aggfunc="mean",
    ).round(2)
    return table.sort_index()

**Public checks — worth 0 points. Run them.**

In [ ]:
table = weather_scoring(games)

assert isinstance(table, pd.DataFrame)
assert list(table.index) == ["Cloudy", "Overcast", "Rain", "Snow", "Sunny", "Windy"]
assert list(table.columns) == ["Away", "Home", "Neutral"]
assert pd.isna(table.loc["Snow", "Neutral"]), "no neutral-site snow games — leave it NaN"
assert abs(table.loc["Snow", "Home"] - 41.5) < 0.01
assert abs(table.loc["Windy", "Away"] - 16.67) < 0.01

print("❄️  Snowy home games are the highest-scoring cell in the table.")
print("🌬️  Windy road games are the lowest. Take that however you like.")
print("✅ Q10 public checks passed — that's all ten. Go Blue. 〽️")
table

---

# 🎉 Victory lap

That's 100 points. Everything below here is **ungraded** — no asserts, no
hidden tests, nothing to submit. It is here because the interesting part of
data work starts once the required questions are done.

Run them, break them, rewrite them. A few starting points:


In [ ]:
# Scoring trend by season, with the coaching changes marked.
card = season_report_card(games)
coaches = games.groupby("season")["head_coach"].first()

summary = card.join(coaches.rename("head_coach"))
summary["ppg"] = (summary["points_for"] / 13).round(1)
summary["opp_ppg"] = (summary["points_against"] / 13).round(1)
summary[["head_coach", "wins", "losses", "win_pct", "ppg", "opp_ppg"]]

In [ ]:
# Which coach had the best simulated run?
scored = add_result_columns(games)
scored.groupby("head_coach").agg(
    seasons=("season", "nunique"),
    games=("won", "size"),
    wins=("won", "sum"),
    win_pct=("won", "mean"),
    avg_margin=("point_diff", "mean"),
).round(3).sort_values("win_pct", ascending=False)

In [ ]:
# Does a bigger crowd come with a bigger win? (Home games only.)
home = add_result_columns(games).query("site == 'Home'").copy()
home["attendance"] = filled_attendance(games)[home.index]
print("correlation between attendance and margin of victory:",
      round(home["attendance"].corr(home["point_diff"]), 3))

home["crowd_bucket"] = pd.qcut(home["attendance"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
home.groupby("crowd_bucket", observed=True)["point_diff"].mean().round(2)

In [ ]:
# Ticket prices vs. how good the opponent was. Does the market know?
merged = games.merge(pd.read_csv("data/opponents.csv"), on="opponent", how="left")
merged["price"] = ticket_prices(games).to_numpy()
merged.groupby("conference")["price"].agg(["count", "mean", "max"]).round(2) \
      .sort_values("mean", ascending=False)

In [ ]:
# A picture, if matplotlib is installed.
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib is not installed — try: pip install matplotlib")
else:
    fig, ax = plt.subplots(figsize=(11, 4))
    card = season_report_card(games)
    ax.bar(card.index, card["points_for"], label="points for", color="#00274C")
    ax.bar(card.index, -card["points_against"], label="points against", color="#FFCB05")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title("Simulated Michigan scoring by season")
    ax.set_xlabel("season")
    ax.set_ylabel("points")
    ax.legend()
    plt.tight_layout()
    plt.show()

---

### Your turn

Add a cell and answer a question nobody asked you to answer. Some that the data
can actually support:

- Is there a **home-field advantage** in this simulation, and how big is it?
- Does Michigan score more in **September** than in **November**? (`.dt.month`)
- What is the **longest simulated winning streak**, and when did it end?
  (harder than it looks — try `.cumsum()` on a boolean)
- Which opponent has been the **most expensive ticket** relative to how good
  they were?
- Do **blowouts** cluster in the non-conference schedule? Should they?

---

<div align="center">

**〽️ Go Blue.**

*Hail to the victors, and to everyone who read the docstring before writing the function.*

</div>
